In [1]:
# Descarga y preparación de datos
# Importación de librerías
import pandas as pd
from pathlib import Path 

In [2]:
# Carga de datos y visualización de las primeras filas
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "churn.csv"

Churn_data = pd.read_csv(DATA_PATH)
Churn_data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


In [3]:
# Información general del DataFrame
Churn_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.2 MB


In [4]:
# Número de filas y columnas
print(f'Número de filas: {Churn_data.shape[0]}, Número de columnas: {Churn_data.shape[1]}')

Número de filas: 10000, Número de columnas: 14


In [5]:
# Descripción del DataFrame
print(Churn_data.describe())

         RowNumber    CustomerId   CreditScore           Age       Tenure  \
count  10000.00000  1.000000e+04  10000.000000  10000.000000  9091.000000   
mean    5000.50000  1.569094e+07    650.528800     38.921800     4.997690   
std     2886.89568  7.193619e+04     96.653299     10.487806     2.894723   
min        1.00000  1.556570e+07    350.000000     18.000000     0.000000   
25%     2500.75000  1.562853e+07    584.000000     32.000000     2.000000   
50%     5000.50000  1.569074e+07    652.000000     37.000000     5.000000   
75%     7500.25000  1.575323e+07    718.000000     44.000000     7.000000   
max    10000.00000  1.581569e+07    850.000000     92.000000    10.000000   

             Balance  NumOfProducts    HasCrCard  IsActiveMember  \
count   10000.000000   10000.000000  10000.00000    10000.000000   
mean    76485.889288       1.530200      0.70550        0.515100   
std     62397.405202       0.581654      0.45584        0.499797   
min         0.000000       1.00000

In [6]:
# Buscamos y contamos valores nulos
Churn_data.isna().sum()

RowNumber            0
CustomerId           0
Surname              0
CreditScore          0
Geography            0
Gender               0
Age                  0
Tenure             909
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Exited               0
dtype: int64

En la columna 'Tenure' tenemos 909 filas sin algún valor, veremos más adelante que haremos con ellos, porque representan el periodo en que estuvieron los clientes con el servicio

In [7]:
# Buscaremos valores duplicados
print(f'Número de filas duplicadas: {Churn_data.duplicated().sum()}')

Número de filas duplicadas: 0


In [8]:
# Separamos identificadores, características y variable objetivo
columnas_identificadoras = ['RowNumber', 'CustomerId', 'Surname']

X = Churn_data.drop(columns=columnas_identificadoras + ['Exited'])
y = Churn_data['Exited']

print(f'Características: {X.shape}')
print(f'Variable objetivo: {y.shape}')
print('\nDistribución de la variable objetivo:')
print(y.value_counts(normalize=True).round(4))

Características: (10000, 10)
Variable objetivo: (10000,)

Distribución de la variable objetivo:
Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64


In [9]:
# Dividimos los datos antes de realizar cualquier transformación
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Entrenamiento: {X_train.shape}')
print(f'Prueba: {X_test.shape}')

print('\nDistribución en entrenamiento:')
print(y_train.value_counts(normalize=True).round(4))

Entrenamiento: (8000, 10)
Prueba: (2000, 10)

Distribución en entrenamiento:
Exited
0    0.7962
1    0.2037
Name: proportion, dtype: float64


In [10]:
# Preparamos las variables numéricas y categóricas sin fuga de información

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

columnas_numericas = X.select_dtypes(include='number').columns.tolist()
columnas_categoricas = X.select_dtypes(exclude='number').columns.tolist()

pipeline_numerico = Pipeline(steps=[
    ('imputacion', SimpleImputer(strategy='median')),
    ('escalado', StandardScaler())
])

pipeline_categorico = Pipeline(steps=[
    ('imputacion', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(
        drop='first',
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocesador = ColumnTransformer(
    transformers=[
        ('numericas', pipeline_numerico, columnas_numericas),
        ('categoricas', pipeline_categorico, columnas_categoricas)
    ],
    verbose_feature_names_out=False
)

X_train_procesado = preprocesador.fit_transform(X_train)
X_test_procesado = preprocesador.transform(X_test)

nombres_caracteristicas = preprocesador.get_feature_names_out()

X_train_procesado = pd.DataFrame(
    X_train_procesado,
    columns=nombres_caracteristicas,
    index=X_train.index
)

X_test_procesado = pd.DataFrame(
    X_test_procesado,
    columns=nombres_caracteristicas,
    index=X_test.index
)

print(f'Entrenamiento procesado: {X_train_procesado.shape}')
print(f'Prueba procesada: {X_test_procesado.shape}')
print(f'Valores nulos restantes: {X_train_procesado.isna().sum().sum()}')
print('\nCaracterísticas finales:')
print(nombres_caracteristicas)

Entrenamiento procesado: (8000, 11)
Prueba procesada: (2000, 11)
Valores nulos restantes: 0

Características finales:
['CreditScore' 'Age' 'Tenure' 'Balance' 'NumOfProducts' 'HasCrCard'
 'IsActiveMember' 'EstimatedSalary' 'Geography_Germany' 'Geography_Spain'
 'Gender_Male']


In [11]:
# Entrenamos un modelo de regresión logística como ejemplo
from sklearn.linear_model import LogisticRegression

modelo_logistico_base = LogisticRegression(max_iter=1000, random_state=42)

modelo_logistico_base.fit(X_train_procesado, y_train)

print('Modelo base entrenado correctamente')

Modelo base entrenado correctamente


In [12]:
# Evaluamos el modelo base

from sklearn.metrics import f1_score, roc_auc_score

y_pred_base = modelo_logistico_base.predict(X_test_procesado)
y_proba_base = modelo_logistico_base.predict_proba(
    X_test_procesado
)[:, 1]

f1_base = f1_score(y_test, y_pred_base)
auc_base = roc_auc_score(y_test, y_proba_base)

print(f'F1 Score — modelo base: {f1_base:.4f}')
print(f'ROC AUC — modelo base: {auc_base:.4f}')

F1 Score — modelo base: 0.2873
ROC AUC — modelo base: 0.7749


El modelo base que entrenamos presenta un F1 score bajo del 0.28, lo que indica un desempeño deficiente en la identificación de clientes que abandonan el banco. Sin embargo, el AUC-ROC de 0.77 sugiere que el modelo tiene una buena capacidad de discriminación entre clases. Esto indica que el problema principal si está en el desbalance de clases y en el umbral de clasificación que estamos utilizando.

In [13]:
# Mejoramos la regresión logística ajustando el peso de las clases

modelo_logistico_balanceado = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

modelo_logistico_balanceado.fit(
    X_train_procesado,
    y_train
)

y_pred_balanceado = modelo_logistico_balanceado.predict(
    X_test_procesado
)

y_proba_balanceado = modelo_logistico_balanceado.predict_proba(
    X_test_procesado
)[:, 1]

print('Modelo logístico balanceado entrenado correctamente.')

Modelo logístico balanceado entrenado correctamente.


In [14]:
# Evaluamos la regresión logística con clases balanceadas

f1_balanceado = f1_score(
    y_test,
    y_pred_balanceado
)

auc_balanceado = roc_auc_score(
    y_test,
    y_proba_balanceado
)

print(f'F1 Score — modelo balanceado: {f1_balanceado:.4f}')
print(f'ROC AUC — modelo balanceado: {auc_balanceado:.4f}')

F1 Score — modelo balanceado: 0.5000
ROC AUC — modelo balanceado: 0.7773


Nos damos cuenta de que corregir el desbalance sí mejora el rendimiento en la clase minoritaria

In [15]:
# Aplicamos SMOTE únicamente sobre el conjunto de entrenamiento procesado

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_procesado,
    y_train
)

print(f'Antes de SMOTE: {X_train_procesado.shape}')
print(f'Después de SMOTE: {X_train_smote.shape}')

print('\nDistribución después de SMOTE:')
print(y_train_smote.value_counts())

Antes de SMOTE: (8000, 11)
Después de SMOTE: (12740, 11)

Distribución después de SMOTE:
Exited
1    6370
0    6370
Name: count, dtype: int64


In [16]:
# Entrenamos una regresión logística con el conjunto balanceado mediante SMOTE

modelo_logistico_smote = LogisticRegression(
    max_iter=1000,
    random_state=42
)

modelo_logistico_smote.fit(
    X_train_smote,
    y_train_smote
)

print('Modelo con SMOTE entrenado correctamente.')


Modelo con SMOTE entrenado correctamente.


In [17]:
# Evaluamos el modelo con SMOTE sobre el conjunto de prueba original

y_pred_smote = modelo_logistico_smote.predict(
    X_test_procesado
)

y_proba_smote = modelo_logistico_smote.predict_proba(
    X_test_procesado
)[:, 1]

f1_smote = f1_score(
    y_test,
    y_pred_smote
)

auc_smote = roc_auc_score(
    y_test,
    y_proba_smote
)

print(f'F1 Score — SMOTE: {f1_smote:.4f}')
print(f'ROC AUC — SMOTE: {auc_smote:.4f}')

F1 Score — SMOTE: 0.5058
ROC AUC — SMOTE: 0.7754


### Resultados con SMOTE

SMOTE se aplicó únicamente sobre el conjunto de entrenamiento procesado, mientras que la evaluación se realizó sobre el conjunto de prueba original para conservar una situación representativa del problema real.

El modelo alcanzó un F1-score de 0.5058 y un ROC AUC de 0.7754. Esto representa una mejora importante frente al modelo base y un desempeño muy similar al modelo de regresión logística con `class_weight='balanced'`.

Aunque SMOTE obtuvo un F1 ligeramente superior, el modelo con pesos balanceados sigue siendo una alternativa más sencilla, ya que no requiere generar observaciones sintéticas.

In [18]:
# Seleccionamos el threshold mediante validación cruzada
# sin utilizar el conjunto de prueba

import numpy as np

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)

pipeline_balanceado_cv = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

validacion_cruzada = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

probabilidades_oof = cross_val_predict(
    pipeline_balanceado_cv,
    X_train,
    y_train,
    cv=validacion_cruzada,
    method='predict_proba'
)[:, 1]

thresholds = np.arange(0.10, 0.90, 0.01)

f1_validacion = [
    f1_score(
        y_train,
        (probabilidades_oof >= threshold).astype(int)
    )
    for threshold in thresholds
]

mejor_threshold = thresholds[np.argmax(f1_validacion)]
mejor_f1_validacion = max(f1_validacion)

# Evaluación final sobre el conjunto de prueba intacto
y_pred_threshold = (
    y_proba_balanceado >= mejor_threshold
).astype(int)

f1_threshold_test = f1_score(
    y_test,
    y_pred_threshold
)

print(f'Mejor threshold en validación: {mejor_threshold:.2f}')
print(f'F1 en validación cruzada: {mejor_f1_validacion:.4f}')
print(f'F1 final en prueba: {f1_threshold_test:.4f}')

Mejor threshold en validación: 0.54
F1 en validación cruzada: 0.4963
F1 final en prueba: 0.5038


### Ajuste del threshold

El threshold se seleccionó mediante validación cruzada sobre el conjunto de entrenamiento, evitando utilizar el conjunto de prueba durante el ajuste.

El mejor umbral encontrado fue 0.54, con un F1-score de 0.4963 en validación cruzada y 0.5038 en el conjunto de prueba. La mejora frente al threshold predeterminado de 0.50 fue marginal, por lo que el ajuste del umbral no produjo un beneficio suficientemente relevante.

A continuación, se evaluará Random Forest para capturar relaciones no lineales entre las características.


## Entrenamiento del modelo Random Forest

Debido a que el modelo de Logistic Regression no logró alcanzar el rendimiento esperado, se decidió probar un modelo más robusto: Random Forest. Este modelo permite capturar relaciones más complejas entre las variables, lo que puede mejorar la predicción de clientes que abandonan el banco.


In [19]:
# Entrenamos Random Forest para capturar relaciones no lineales

from sklearn.ensemble import RandomForestClassifier

modelo_rf_base = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

modelo_rf_base.fit(
    X_train_procesado,
    y_train
)

print('Random Forest base entrenado correctamente.')

Random Forest base entrenado correctamente.


In [20]:
# Generamos predicciones con Random Forest base

y_pred_rf_base = modelo_rf_base.predict(
    X_test_procesado
)

y_proba_rf_base = modelo_rf_base.predict_proba(
    X_test_procesado
)[:, 1]

In [21]:
# Evaluamos Random Forest base

f1_rf_base = f1_score(
    y_test,
    y_pred_rf_base
)

auc_rf_base = roc_auc_score(
    y_test,
    y_proba_rf_base
)

print(f'F1 Score — Random Forest base: {f1_rf_base:.4f}')
print(f'ROC AUC — Random Forest base: {auc_rf_base:.4f}')

F1 Score — Random Forest base: 0.6103
ROC AUC — Random Forest base: 0.8558


In [22]:
# Ajustamos Random Forest mediante validación cruzada

from sklearn.model_selection import RandomizedSearchCV

pipeline_rf = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=1
    ))
])

parametros_rf = {
    'modelo__n_estimators': [200, 300, 400],
    'modelo__max_depth': [8, 10, 12, None],
    'modelo__min_samples_split': [2, 5, 10],
    'modelo__min_samples_leaf': [1, 2, 4],
    'modelo__max_features': ['sqrt', 'log2']
}

busqueda_rf = RandomizedSearchCV(
    estimator=pipeline_rf,
    param_distributions=parametros_rf,
    n_iter=12,
    scoring='f1',
    cv=validacion_cruzada,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

busqueda_rf.fit(
    X_train,
    y_train
)

modelo_rf_ajustado = busqueda_rf.best_estimator_

print('Mejores parámetros:')
print(busqueda_rf.best_params_)

print(
    f'\nMejor F1 en validación cruzada: '
    f'{busqueda_rf.best_score_:.4f}'
)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Mejores parámetros:
{'modelo__n_estimators': 200, 'modelo__min_samples_split': 5, 'modelo__min_samples_leaf': 1, 'modelo__max_features': 'log2', 'modelo__max_depth': 12}

Mejor F1 en validación cruzada: 0.6219


In [23]:
# Evaluamos el mejor Random Forest sobre el conjunto de prueba intacto

y_pred_rf_ajustado = modelo_rf_ajustado.predict(
    X_test
)

y_proba_rf_ajustado = modelo_rf_ajustado.predict_proba(
    X_test
)[:, 1]

f1_rf_ajustado = f1_score(
    y_test,
    y_pred_rf_ajustado
)

auc_rf_ajustado = roc_auc_score(
    y_test,
    y_proba_rf_ajustado
)

print(
    f'F1 Score — Random Forest ajustado: '
    f'{f1_rf_ajustado:.4f}'
)

print(
    f'ROC AUC — Random Forest ajustado: '
    f'{auc_rf_ajustado:.4f}'
)

F1 Score — Random Forest ajustado: 0.6219
ROC AUC — Random Forest ajustado: 0.8610


In [24]:
# Comparamos el rendimiento de todos los modelos evaluados

resultados_modelos = pd.DataFrame({
    'Modelo': [
        'Regresión logística base',
        'Logística balanceada',
        'Logística balanceada + threshold',
        'Logística con SMOTE',
        'Random Forest base',
        'Random Forest ajustado'
    ],
    'F1 Score': [
        f1_base,
        f1_balanceado,
        f1_threshold_test,
        f1_smote,
        f1_rf_base,
        f1_rf_ajustado
    ],
    'ROC AUC': [
        auc_base,
        auc_balanceado,
        auc_balanceado,
        auc_smote,
        auc_rf_base,
        auc_rf_ajustado
    ]
})

resultados_modelos = (
    resultados_modelos
    .sort_values('F1 Score', ascending=False)
    .reset_index(drop=True)
)

resultados_modelos.style.format({
    'F1 Score': '{:.4f}',
    'ROC AUC': '{:.4f}'
})

,Modelo,F1 Score,ROC AUC
0,Random Forest ajustado,0.6219,0.8610
1,Random Forest base,0.6103,0.8558
2,Logística con SMOTE,0.5058,0.7754
3,Logística balanceada + threshold,0.5038,0.7773
4,Logística balanceada,0.5000,0.7773
5,Regresión logística base,0.2873,0.7749


In [25]:
# Visualizamos la comparación de desempeño entre modelos

import plotly.express as px

resultados_largos = resultados_modelos.melt(
    id_vars='Modelo',
    value_vars=['F1 Score', 'ROC AUC'],
    var_name='Métrica',
    value_name='Puntuación'
)

fig_comparacion = px.bar(
    resultados_largos,
    y='Modelo',
    x='Puntuación',
    color='Métrica',
    orientation='h',
    barmode='group',
    text='Puntuación',
    category_orders={
        'Modelo': resultados_modelos['Modelo'].tolist()
    },
    color_discrete_map={
        'F1 Score': '#2563EB',
        'ROC AUC': '#D97706'
    },
    title='Comparación de desempeño por modelo'
)

fig_comparacion.update_traces(
    texttemplate='%{text:.3f}',
    textposition='outside',
    cliponaxis=False
)

fig_comparacion.update_layout(
    template='plotly_white',
    height=500,
    xaxis_title='Puntuación',
    yaxis_title='',
    legend_title='',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='left',
        x=0
    ),
    margin=dict(l=220, r=50, t=100, b=50)
)

fig_comparacion.update_xaxes(
    range=[0, 1],
    showgrid=True,
    gridcolor='#E5E7EB'
)

fig_comparacion.update_yaxes(
    autorange='reversed'
)

fig_comparacion.show()

## Exportación del modelo

El pipeline final se guarda para reutilizarlo en la aplicación de Streamlit. El archivo integra tanto el preprocesamiento como el modelo Random Forest ajustado.

In [28]:
# Exportamos el modelo y las métricas para el dashboard

from joblib import dump

MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / 'churn_pipeline.joblib'
METRICS_PATH = MODELS_DIR / 'model_metrics.csv'

dump(
    modelo_rf_ajustado,
    MODEL_PATH
)

resultados_modelos.to_csv(
    METRICS_PATH,
    index=False
)

print(f'Modelo guardado en: {MODEL_PATH}')
print(f'Métricas guardadas en: {METRICS_PATH}')

Modelo guardado en: c:\Users\cib3r\Desktop\SCHOOL\Proyecto Triple10\Sprint 11\Proyecto_s11\Proyect_S11\Proyect_S11\models\churn_pipeline.joblib
Métricas guardadas en: c:\Users\cib3r\Desktop\SCHOOL\Proyecto Triple10\Sprint 11\Proyecto_s11\Proyect_S11\Proyect_S11\models\model_metrics.csv


## Conclusión final

Se evaluaron distintos enfoques para predecir la pérdida de clientes del banco: regresión logística, ajuste de pesos de clase, sobremuestreo mediante SMOTE y Random Forest.

La regresión logística base obtuvo un F1-score bajo de 0.2873. Al considerar el desbalance de clases, el F1 aumentó a 0.5000 con `class_weight='balanced'` y a 0.5058 mediante SMOTE.

Random Forest presentó un rendimiento considerablemente superior. El modelo base alcanzó un F1-score de 0.6103 y un ROC AUC de 0.8558. Posteriormente, los hiperparámetros se ajustaron mediante validación cruzada, sin utilizar el conjunto de prueba durante la selección.

El modelo final obtuvo un F1-score de 0.6219 y un ROC AUC de 0.8610 sobre el conjunto de prueba intacto. Por ello, Random Forest ajustado fue seleccionado como el mejor modelo para este problema.

Además, el modelo final integra el preprocesamiento y la clasificación dentro de un único `Pipeline`, permitiendo recibir registros con la estructura original del dataset. Esto facilita su reutilización y posterior integración en una aplicación interactiva.

Aunque el modelo ofrece una buena capacidad para distinguir clientes con diferente riesgo de abandono, una implementación empresarial debería considerar también el costo de las acciones de retención, el valor de cada cliente y la validación del modelo con información más reciente.
